<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_82_linux_bash_for_ml_engineers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🐧 **Module 82 — Linux + Bash for ML Engineers** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 11 · Foundations Backfill


# Module 82 — Linux + Bash for ML Engineers

> Every ML/AI engineer eventually SSHes into a GPU box at 2 AM because a vLLM pod won't start, training stalled at step 12 of 50 000, or `nvidia-smi` shows three GPUs missing. **The course assumed you knew Linux.** This module is the **practical map of the OS** an ML engineer touches — `/proc`, `nvidia-smi` workflows, `journalctl`, `cgroups`, `tmux`, `rsync`, `perf`, `eBPF` — without the system-administration bloat.
>
> Every command here is one you'd type in a real incident or training-prep flow.

### What you'll cover
1. The filesystem map an ML engineer needs (where things actually live)
2. Process management — `ps`, `top`, `htop`, `lsof`, `strace`
3. **`nvidia-smi` workflows** that matter
4. **systemd** for long-running services
5. **`/proc` and `/sys`** — the kernel's API
6. **cgroups + namespaces** — the Docker primitives
7. Logs — `journalctl`, `dmesg`, `logrotate`, structured logging
8. Networking — `ss`, `tcpdump`, `iperf3`, `mtr`
9. Disk + memory — `iostat`, `fio`, `vmstat`, `smem`, `free`
10. **Bash for ML ops** — the patterns that survive contact with production
11. **`perf` + eBPF** — the production performance toolkit


## 1 · The filesystem map for ML engineers

```
   /
   ├── etc/                 # config files
   │   ├── systemd/          system service definitions
   │   └── nvidia/           NVIDIA driver config
   ├── usr/
   │   ├── local/cuda/        CUDA toolkit (compiler, libs, headers)
   │   └── local/nvidia/      driver runtime libs
   ├── var/
   │   ├── log/              system logs (journalctl tails these)
   │   └── lib/docker/        container layers (huge — watch fill-up)
   ├── proc/                 # kernel virtual FS — process info, GPU info
   │   ├── meminfo            memory facts
   │   ├── cpuinfo            CPU facts
   │   ├── interrupts         IRQ counters
   │   └── 1234/              one dir per running PID (status, fd, etc.)
   ├── sys/                  # kernel objects — devices, fans, NUMA
   │   ├── class/net/         network interfaces
   │   ├── class/drm/         GPU device tree
   │   ├── bus/pci/devices/   every PCIe card (incl. GPUs and NICs)
   │   └── devices/system/cpu CPU topology + scaling governor
   ├── dev/                  # device nodes
   │   ├── nvidia0, nvidia1, …  one per GPU
   │   ├── nvidia-uvm           unified memory driver
   │   └── nvme0n1, nvme1n1     NVMe devices
   ├── home/                 # user homes
   ├── mnt/, /opt/, /srv/    # mounts for shared FS (Lustre, NFS — M78)
   └── tmp/, /run/, /scratch # scratch + per-boot files
```

Three things to commit to memory:
- `/proc/$$/...` is *your shell's* runtime info; `/proc/<PID>/...` is any other process.
- `/dev/nvidia*` is the **only** file path that grants a container access to a GPU.
- `/var/log/` fills up. **Check it first** when nodes mysteriously OOM or die.

## 2 · Process management — the everyday tools

| Command | What |
|---|---|
| `ps aux` | every process, full args, RSS, %CPU |
| `ps -eo pid,ppid,pcpu,pmem,etime,cmd --sort=-pcpu \| head` | top CPU consumers, sortable |
| `top -c` / `htop` | live view; `htop` is the friendlier UI |
| `pidstat -u -p <pid> 1` | per-PID CPU at 1s intervals |
| `kill -9 <pid>` / `killall -9 <name>` | force-kill |
| `pgrep -fa python` | regex search by command line |
| `lsof -p <pid>` | every file (sockets, mmaps, devices) a process has open |
| `lsof -i :8000` | "who has port 8000?" — *the* `EADDRINUSE` debugger |
| `strace -p <pid> -f -tt -T -s 256` | syscall trace; "what is it actually doing?" |
| `nsenter -t <pid> -a` | enter the namespaces of another process — debug containers |

> 🟡 **`strace` is the truth.** When a Python service "hangs" but isn't using CPU, attach `strace -p $(pgrep -fa app.py)`. The repeating syscall (often a `read()` on a socket) tells you who it's waiting for.

In [ ]:
# Example shell commands you'd type during a real incident
incident_shell = '''
# What's eating the CPU?
ps -eo pid,ppid,user,pcpu,pmem,etime,cmd --sort=-pcpu | head -20

# Which python process owns port 8000?
sudo lsof -nP -iTCP:8000 -sTCP:LISTEN

# Why is this process stuck? (won't return CPU but won't exit)
sudo strace -p 12345 -f -tt -T -s 256 -o /tmp/trace.txt &
sleep 5 ; kill %1
tail /tmp/trace.txt        # look for repeating syscall

# Memory + open-file pressure
sudo cat /proc/12345/status | grep -E "(VmRSS|VmPeak|Threads|FDSize)"
sudo ls -l /proc/12345/fd | wc -l        # currently open FDs

# Why is the box itself slow?
uptime ; free -h ; vmstat 1 5
'''
print(incident_shell)

## 3 · `nvidia-smi` workflows that matter

In [ ]:
nvidia_smi_recipes = '''
# 1. Watch GPUs live (every 1 sec)
nvidia-smi -l 1

# 2. Per-process GPU usage — who owns what memory
nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

# 3. Topology matrix — which GPUs see each other over NVLink (M77)
nvidia-smi topo -m

# 4. Persistence mode — needed before fabric-manager / NCCL is happy
sudo nvidia-smi -pm 1

# 5. Power & clock limits (e.g. cap to 350W during a benchmark sweep)
sudo nvidia-smi -i 0 -pl 350
sudo nvidia-smi -i 0 -lgc 1500,1700           # lock graphics clock to a range

# 6. ECC errors (silent killer — check daily)
nvidia-smi --query-gpu=ecc.errors.uncorrected.aggregate.total --format=csv

# 7. Xid messages — diagnostics in dmesg
sudo dmesg -T | grep -i "NVRM: Xid"

# 8. Useful aliases (put in ~/.bashrc)
alias gpu='nvidia-smi -l 1'
alias gpus='nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw --format=csv,noheader'
'''
print(nvidia_smi_recipes)

**The four `nvidia-smi` outputs every on-call engineer should be able to read in three seconds:**
1. **GPU utilization %** — if it's stuck at ~0% during training, your model is CPU-bound or the data pipeline is the bottleneck.
2. **Memory used/total** — if you're at 95 % during training and OOM'ing, it's time to reduce batch size or enable gradient checkpointing.
3. **Power draw vs. limit** — if it's well below TDP, you're not actually compute-bound.
4. **`Xid` messages in `dmesg`** — these are the canonical hardware error codes. **Xid 79 = "GPU has fallen off the bus"** (M78).

## 4 · systemd — every long-running service

In [ ]:
systemd_unit = '''
# /etc/systemd/system/vllm-server.service
[Unit]
Description=vLLM inference server
Wants=network-online.target nvidia-persistenced.service
After=network-online.target nvidia-persistenced.service

[Service]
Type=simple
User=mlops
Group=mlops
WorkingDirectory=/opt/vllm
ExecStart=/usr/bin/python -m vllm.entrypoints.openai.api_server --model meta-llama/Llama-3.3-70B-Instruct --tensor-parallel-size 8
Restart=always
RestartSec=10
TimeoutStartSec=600        # big models take time to load
KillSignal=SIGTERM
TimeoutStopSec=60
LimitNOFILE=1048576        # open-file limit
LimitMEMLOCK=infinity       # required for CUDA + RDMA
Environment="CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7"
Environment="HF_HOME=/data/hf-cache"

# Hardening
NoNewPrivileges=true
PrivateTmp=true

[Install]
WantedBy=multi-user.target
'''
print(systemd_unit)

Commands you'll type 100 times:

```bash
sudo systemctl daemon-reload
sudo systemctl enable --now vllm-server.service
sudo systemctl status vllm-server.service
sudo systemctl restart vllm-server.service
journalctl -u vllm-server.service -f --since "10 min ago"   # live log
journalctl -u vllm-server.service -p err --since today      # errors only
```

> 🟡 Five `[Service]` settings ML engineers actually care about: `Restart=always`, `TimeoutStartSec` (big model loads), `LimitMEMLOCK=infinity` (CUDA + RDMA require it), `LimitNOFILE` (sockets per worker), `Environment=` (cleaner than wrapper scripts).

## 5 · `/proc` + `/sys` — the kernel's API

In [ ]:
proc_sys_recipes = '''
# Memory facts (used by top, free, etc.)
cat /proc/meminfo | head -5

# CPU topology
lscpu                                       # nicer view
cat /proc/cpuinfo | grep -m1 "model name"

# Per-process info — same shape across every PID
cat /proc/<pid>/status                      # VmRSS, threads, cgroups
cat /proc/<pid>/cmdline | tr '\0' ' '       # the full launch command
ls -l /proc/<pid>/fd | head                 # which FDs are open
cat /proc/<pid>/limits                      # ulimits actually in effect

# CPU governor & turbo
cat /sys/devices/system/cpu/cpu0/cpufreq/scaling_governor
echo performance | sudo tee /sys/devices/system/cpu/cpu*/cpufreq/scaling_governor

# NUMA topology (matters for GPU-NIC alignment — M77)
numactl --hardware
cat /sys/devices/system/node/online

# PCIe topology (GPUs + NICs)
lspci | grep -i -E "(nvidia|mellanox|broadcom)"
lspci -vvv -s <bus_id> | grep LnkSta        # actual link speed/width

# Network interfaces & speed
ip -br addr
ethtool eth0 | grep -i speed
'''
print(proc_sys_recipes)

**Why this matters.** `nvidia-smi` shows GPU state; `/proc` and `/sys` show the **rest of the box**. When a training job is slow despite hot GPUs, the answer is in CPU governor (`powersave`), NUMA misalignment, or a flapping PCIe link — none of which `nvidia-smi` can tell you.

## 6 · cgroups + namespaces — the Docker primitives

Every container is just a process inside a **cgroup** (resource limits) and a **set of namespaces** (isolation). Knowing this debugs every Docker / k8s issue.

```
   container (e.g. a Docker / k8s pod) == process(es)
                                            +
                                         cgroups          # what it can USE
                                            +
                                         namespaces       # what it can SEE
```

Six namespaces matter for ML:
- **PID** — what processes does it see?
- **NET** — its own network stack
- **MNT** — its own filesystem mounts
- **IPC** — its own shared memory (`/dev/shm`)
- **UTS** — its own hostname
- **USER** — UID mapping (rootless containers)

In [ ]:
cgroup_recipes = '''
# Inspect cgroups for a container's PID
PID=$(docker inspect -f "{{.State.Pid}}" my-vllm)
cat /proc/$PID/cgroup                     # what cgroups it belongs to

# Apply a memory limit live (cgroups v2)
sudo systemctl set-property --runtime my.slice MemoryMax=8G

# Enter a container's namespaces (debug without exec'ing into the container)
sudo nsenter -t $PID -a                  # full set
sudo nsenter -t $PID -n -- ss -tlnp     # listening sockets inside the container

# Check what a container actually sees
docker exec -it my-vllm sh -c "cat /proc/self/cgroup; ls /dev/nvidia*"

# Map a host PID to a container
docker ps -q | xargs -I {} docker inspect -f "{{.Name}} {{.State.Pid}}" {}
'''
print(cgroup_recipes)

> 🚨 **The famous `/dev/shm` trap.** PyTorch DataLoaders use shared memory for worker IPC. Docker defaults to **64 MB** of `/dev/shm`. Result: random `dataloader killed worker (pid X)` errors. Fix: `docker run --shm-size=8g ...` or `--ipc=host`. In k8s: `volumes: [{name: dshm, emptyDir: {medium: Memory, sizeLimit: 8Gi}}]`.

## 7 · Logs — `journalctl`, `dmesg`, structured logging

In [ ]:
logs_recipes = '''
# System / service logs — the only command you really need
journalctl -u my-service.service -f                       # tail live
journalctl -u my-service.service --since "1 hour ago"     # recent
journalctl -u my-service.service -p err --since today     # errors only
journalctl -k --since boot                                # kernel ring buffer
journalctl --disk-usage                                   # how big are journals?

# Kernel ring buffer — Xid, OOM, link-down events
sudo dmesg -T | tail -100
sudo dmesg -wH                                            # follow with human time

# OOM-killer events (training OOM, vLLM pods OOM, …)
sudo dmesg -T | grep -i "killed process"

# Logrotate config
ls /etc/logrotate.d/

# Quick check: who's logging the most?
sudo journalctl --disk-usage
sudo du -sh /var/log/* | sort -h | tail
'''
print(logs_recipes)

**Application log discipline ML engineers should adopt:**
1. Log JSON to stdout (not files); let `journald` / Docker / k8s do the rest.
2. Include `trace_id`, `tenant_id`, `model_id`, `request_id` on every line — pair with OTel (M51).
3. Use **structured logging** (`structlog`, `loguru`) so `jq` can slice the file.
4. Different log streams per pod — never log to shared NFS (write amplification kills throughput).

## 8 · Networking — the tools you'll need in 2 AM order

```bash
# Who is listening on which port?
ss -tlnp                                # all TCP listeners + owning PID
ss -tunap state established | head      # active connections

# DNS / connectivity
dig +short api.openai.com               # DNS
curl -v https://api.openai.com/v1/models  # full HTTP
nc -zv host 8000                        # is port open?

# Routing & MTU
ip route
ping -c 5 -M do -s 1472 host           # MTU probe (Don't Fragment)

# Trace path / packet loss
mtr -rwc 30 host                        # the killer combo of ping + traceroute

# Bandwidth between two boxes
# server side:    iperf3 -s
# client side:    iperf3 -c <server-ip> -t 30

# Packet capture
sudo tcpdump -i eth0 -nn -s0 -w /tmp/cap.pcap host 1.2.3.4 and port 8000
```

> 🟡 **`mtr` is the under-used one.** When latency spikes between your training pod and the parallel FS (M78), `mtr -rwc 30 lustre-mds` over a few minutes shows you exactly which hop is dropping packets. `ping` and `traceroute` separately won't.

## 9 · Disk + memory

In [ ]:
disk_mem_recipes = '''
# Disk usage / space remaining
df -hT                                  # mounted filesystems
du -sh /var/log/* | sort -h | tail      # what's eating the disk
df -i                                   # inodes (small-file storms exhaust these)

# Disk IO live
iostat -x 1                              # extended; %util, await, w/s, r/s
iotop -oP                                # which process is hitting disk hardest
sudo dstat -tdngy 1                      # mixed system view

# Synthetic disk benchmark (M78 storage validation)
sudo fio --name=randwrite --filename=/scratch/fio.test --rw=randwrite \
         --bs=4k --size=4G --iodepth=64 --numjobs=4 \
         --time_based --runtime=30 --group_reporting

# Memory + swap
free -h
vmstat 1 10                              # paging, context switches, runqueue
smem -tk                                 # PSS — proportional set size (multi-process picture)

# Per-process memory breakdown
sudo pmap -x <pid> | tail
cat /proc/<pid>/status | grep -E "(VmRSS|VmHWM|VmSwap|RssAnon|RssFile|VmData)"
'''
print(disk_mem_recipes)

**Two ML-specific memory traps:**
- **`HF_HOME`** silently downloads multi-hundred-GB checkpoints to whatever drive you set it to. Move it to a fast big disk.
- **Python multiprocessing** (DataLoaders) shows **N× the RSS** of the model. Each worker forks; without `--ipc=host` you'll fill `/dev/shm`.

## 10 · Bash for ML ops — the patterns that survive contact with prod

In [ ]:
bash_patterns = r'''
#!/usr/bin/env bash
# train-a-model.sh — the production-grade shape

# Strict mode — fail fast on errors / undefined vars / pipe failures
set -Eeuo pipefail

# Trap errors with a helpful message
trap 'rc=$?; echo "FAILED line $LINENO: $BASH_COMMAND  (rc=$rc)" >&2; exit $rc' ERR

# Always-run cleanup (works even on Ctrl+C)
cleanup() {
    rc=$?
    rm -rf "${TMPDIR:-/tmp}/build.$$"
    [[ $rc -ne 0 ]] && echo "cleaning up after failure ($rc)" >&2
}
trap cleanup EXIT

# Defaults via parameter expansion
MODEL="${MODEL:-meta-llama/Llama-3.3-70B-Instruct}"
EPOCHS="${EPOCHS:-3}"
SAVE_DIR="${SAVE_DIR:-/scratch/runs/$(date +%F)}"
DRY_RUN="${DRY_RUN:-0}"

# Logging helpers
log()  { printf "[%s] %s
" "$(date +%T)" "$*"; }
die()  { log "ERROR: $*" >&2; exit 1; }

# Pre-flight checks
command -v python >/dev/null || die "python not in PATH"
[[ -d /opt/cuda ]] || die "/opt/cuda missing"
nvidia-smi -L | grep -q UUID || die "no GPUs visible"

mkdir -p "$SAVE_DIR" || die "cannot create $SAVE_DIR"

log "training $MODEL for $EPOCHS epochs into $SAVE_DIR"

# Loop with safe IFS handling (newlines + tabs only)
for shard in $(ls /data/shards/*.parquet); do
    [[ "$DRY_RUN" == "1" ]] && { log "DRY: $shard"; continue; }
    python train.py --shard "$shard" --epochs "$EPOCHS" --out "$SAVE_DIR"         2>&1 | tee -a "$SAVE_DIR/train.log"
done

log "done."
'''
print(bash_patterns)

**The seven habits of effective bash scripts:**

1. **`set -Eeuo pipefail`** — fail fast, fail loud.
2. **`trap ERR` with line-number** — pinpoints the failure.
3. **`trap EXIT` for cleanup** — always-run, even on `Ctrl+C`.
4. **`"${VAR:-default}"`** — defaults with parameter expansion.
5. **Wrap commands in helpers** (`log`, `die`) — DRY + structured stderr.
6. **Pre-flight every dependency** — `command -v`, mount points, GPU presence.
7. **`tee` long-running output to a log file** — keep the terminal but persist.

> 🟡 **What replaces bash above ~200 lines.** Use **Python with `typer` / `click`** as soon as the script has multiple conditional branches, JSON parsing, or retries. Bash is unbeatable for piping CLIs together; it gets brittle once you have control flow.

## 11 · `perf` + eBPF — the production performance toolkit

In [ ]:
perf_ebpf = '''
# Where is CPU time going for a running PID?
sudo perf top -p <pid>                                 # live "hot functions"
sudo perf record -F 99 -p <pid> -g -- sleep 30          # 30s sample
sudo perf report --stdio                                # browse

# Generate a flamegraph (Brendan Gregg's tool)
git clone https://github.com/brendangregg/FlameGraph
sudo perf record -F 99 -p <pid> -g -- sleep 30
sudo perf script | FlameGraph/stackcollapse-perf.pl | FlameGraph/flamegraph.pl > flame.svg

# eBPF / bpftrace — far gentler perf hits, much richer signals
sudo bpftrace -e "tracepoint:syscalls:sys_enter_openat { @[comm] = count(); }"
sudo bpftrace -e "kprobe:do_unlinkat { printf(\"%s deletes %s\\n\", comm, str(arg1)); }"

# bcc-tools — pre-baked one-liners; install once
sudo apt install bpfcc-tools
sudo execsnoop-bpfcc                # every exec() in the system
sudo opensnoop-bpfcc                # every open() — see who is hitting the FS
sudo runqlat-bpfcc 10 1              # scheduler run-queue latency
sudo profile-bpfcc -p <pid> 30      # cheaper alternative to perf record

# NVIDIA Nsight Systems for GPU profiles (ties to M41)
sudo nsys profile -t cuda,nvtx -o profile.qdrep python train.py
'''
print(perf_ebpf)

**Mental model for production performance work:**
- **`perf top`** — first 30 seconds; where's CPU going?
- **Flamegraphs** — what's the call tree underneath those hot functions?
- **bcc / bpftrace** — kernel-side; who's syscalling, who's waiting, who's blocking?
- **Nsight Systems** — GPU side; CUDA kernels + memory transfers (M41).

That four-tool ladder solves 95 % of "why is this slow" tickets.

## ✅ Recap

- The filesystem map: `/proc` for process info, `/sys` for hardware, `/dev/nvidia*` for GPU access, `/var/log` is what fills up first.
- **`nvidia-smi -l 1`**, `topo -m`, the `Xid` grep, and `query-compute-apps` are the four GPU recipes.
- **`systemd` units** with `Restart=always`, `LimitMEMLOCK=infinity`, and `TimeoutStartSec=600` are how every production model service runs.
- **cgroups + namespaces** = containers. Know **`/dev/shm`** for PyTorch DataLoaders.
- `journalctl -u <svc> -f` + `dmesg` get you 90 % of the way into any incident.
- `ss`, `mtr`, `iperf3`, `tcpdump` are the networking 2-AM kit.
- `iostat`, `iotop`, `fio`, `free`, `vmstat`, `smem`, `pmap` for disk + memory.
- Bash with **strict mode + traps + parameter defaults + logging helpers** survives production.
- **`perf` + flamegraphs + bcc/bpftrace + Nsight Systems** is the performance ladder.

Next: **M83 — CNNs Deep Dive (LeNet → ResNet → EfficientNet → YOLO / DETR)**.
